# 5-27 Check Result Analysis

這份 notebook 會把 `5-27 check_result/` 裡所有 `.checked.dot` 的 Approved List 與 Rejected List 抓出來，並檢查它們合併後是否涵蓋完整的 `ontology/ontology_20.dot`。

In [48]:
from pathlib import Path
import re

import pandas as pd

# 設定分析會用到的專案路徑，讓 notebook 從專案根目錄或其他工作目錄開啟時都能找到資料。
PROJECT_ROOT = Path.cwd()
CHECK_DIR = PROJECT_ROOT / "5-27 check_result"
ONTOLOGY_PATH = PROJECT_ROOT / "ontology checker" / "ontology_20.dot"

CHECK_DIR, ONTOLOGY_PATH

(WindowsPath('d:/KaseOnto/5-27 check_result'),
 WindowsPath('d:/KaseOnto/ontology checker/ontology_20.dot'))

In [50]:
# 解析標準 DOT edge，例如："source" -> "target" [label="sense"]。
EDGE_RE = re.compile(
    r'^\s*"(?P<source>.*?)"\s*->\s*"(?P<target>.*?)"\s*\[label="(?P<relation>.*?)"\]\s*$'
)


def edge_key(source, target, relation):
    # 統一 edge 的比較格式，後面會用它檢查 approved/rejected 是否覆蓋 ontology。
    return (source, target, relation)


def parse_dot_edges(text):
    # 從 DOT 主體讀出完整 ontology edge；comment 中的 list 也會符合格式，所以呼叫端要先切好範圍。
    edges = []
    for line in text.splitlines():
        match = EDGE_RE.match(line.strip())
        if match:
            edges.append(edge_key(match["source"], match["target"], match["relation"]))
    return edges


def parse_comment_list(text, header):
    # 只解析指定的 KASEONTO Approved List / Rejected List comment 區塊，不讀取 state 或 DOT 主體。
    lines = text.splitlines()
    start = None
    for index, line in enumerate(lines):
        if line.strip() == f"// KASEONTO {header}":
            start = index + 1
            break
    if start is None:
        return []

    edges = []
    for line in lines[start:]:
        stripped = line.strip()
        if stripped.startswith("// KASEONTO "):
            break
        if stripped == "// empty":
            continue
        if stripped.startswith("// "):
            match = EDGE_RE.match(stripped[3:].strip())
            if match:
                edges.append({
                    "source": match["source"],
                    "target": match["target"],
                    "relation": match["relation"],
                    "rejectReason": "",
                })
    return edges


def parse_checked_file(path):
    # 讀取單一 checked DOT 檔，只抓 Approved List 與 Rejected List 兩個 comment 區塊。
    text = path.read_text(encoding="utf-8")
    rows = []

    source_rows = []
    for row in parse_comment_list(text, "Approved List"):
        source_rows.append({**row, "decision": "approved"})
    for row in parse_comment_list(text, "Rejected List"):
        source_rows.append({**row, "decision": "rejected"})

    for row in source_rows:
        decision = row.get("decision", "").strip().lower()
        if decision not in {"approved", "rejected"}:
            continue
        rows.append({
            "file": path.name,
            "source": row["source"],
            "target": row["target"],
            "relation": row["relation"],
            "decision": decision,
            "reject_reason": row.get("rejectReason", ""),
            "edge_key": edge_key(row["source"], row["target"], row["relation"]),
        })
    return rows

In [51]:
# 載入完整 ontology 與所有 5-27 checked 檔案。
ontology_text = ONTOLOGY_PATH.read_text(encoding="utf-8")
raw_ontology_edges = set(parse_dot_edges(ontology_text))

# analysis 要對齊 ontology checker 的 Check List：root 的 top sense 骨架不進入人工 review。
ontology_sources = {source for source, _target, _relation in raw_ontology_edges}
ontology_targets = {target for _source, target, _relation in raw_ontology_edges}
ontology_root = sorted(ontology_sources - ontology_targets)[0]
excluded_top_sense_edges = {
    edge for edge in raw_ontology_edges if edge[0] == ontology_root and edge[2] == "top sense"
}
ontology_edges = raw_ontology_edges - excluded_top_sense_edges
checked_paths = sorted(CHECK_DIR.glob("*.checked.dot"))

all_rows = []
for path in checked_paths:
    all_rows.extend(parse_checked_file(path))

decisions_df = pd.DataFrame(all_rows)

print(f"raw ontology edge count: {len(raw_ontology_edges)}")
print(f"excluded root top sense count: {len(excluded_top_sense_edges)}")
print(f"reviewable ontology edge count: {len(ontology_edges)}")
print(f"checked file count: {len(checked_paths)}")
print(f"decision row count: {len(decisions_df)}")

raw ontology edge count: 190
excluded root top sense count: 5
reviewable ontology edge count: 185
checked file count: 11
decision row count: 654


In [52]:
# 檢查每個檔案各自的 approved/rejected 數量，以及是否有不屬於原 ontology 的 edge。
per_file_summary = (
    decisions_df.assign(in_ontology=decisions_df["edge_key"].isin(ontology_edges))
    .pivot_table(
        index="file",
        columns="decision",
        values="edge_key",
        aggfunc="nunique",
        fill_value=0,
    )
    .reset_index()
)

for column in ["approved", "rejected"]:
    if column not in per_file_summary.columns:
        per_file_summary[column] = 0

per_file_extra = (
    decisions_df.assign(in_ontology=decisions_df["edge_key"].isin(ontology_edges))
    .groupby("file", as_index=False)
    .agg(total_decided=("edge_key", "nunique"), not_in_ontology=("in_ontology", lambda values: int((~values).sum())))
)

per_file_summary = (
    per_file_summary.merge(per_file_extra, on="file", how="left")
    .sort_values("file")
    [["file", "approved", "rejected", "total_decided", "not_in_ontology"]]
)

per_file_summary

,file,approved,rejected,total_decided,not_in_ontology
0,于皓-ontology_20.checked-10.checked.dot,59,1,60,0
1,梁庭嘉-ontology_20.checked-5.checked.dot,56,4,60,0
2,莊書豪-ontology_20.checked-8.checked.dot,59,1,60,0
3,蔡承軒-ontology_20.checked-11.checked.dot,55,4,59,0
4,蘇于庭-ontology_20.checked-2.checked.dot,57,3,60,0
5,謝瑋庭-ontology_20.checked-4.checked.dot,57,3,60,0
6,郭瀞淇-ontology_20.checked-3.checked.dot,53,7,60,0
7,陳奕儒-ontology_20.checked.dot,60,0,60,0
8,陳宏瑜-ontology_20.checked-7.checked.dot,46,9,55,0
9,陳智偉-ontology_20.checked-6.checked.dot,60,0,60,0


In [53]:
# 合併所有檔案的 Approved List 與 Rejected List，檢查總和是否涵蓋 reviewable ontology。
decided_edges = set(decisions_df["edge_key"])
approved_edges = set(decisions_df.loc[decisions_df["decision"] == "approved", "edge_key"])
rejected_edges = set(decisions_df.loc[decisions_df["decision"] == "rejected", "edge_key"])

missing_edges = ontology_edges - decided_edges
extra_edges = decided_edges - ontology_edges
conflicting_edges = approved_edges & rejected_edges

coverage_summary = pd.DataFrame([
    {"metric": "raw_ontology_edges", "count": len(raw_ontology_edges)},
    {"metric": "excluded_root_top_sense_edges", "count": len(excluded_top_sense_edges)},
    {"metric": "reviewable_ontology_edges", "count": len(ontology_edges)},
    {"metric": "unique_approved_edges", "count": len(approved_edges)},
    {"metric": "unique_rejected_edges", "count": len(rejected_edges)},
    {"metric": "unique_decided_edges", "count": len(decided_edges)},
    {"metric": "missing_from_approved_or_rejected", "count": len(missing_edges)},
    {"metric": "extra_not_in_ontology", "count": len(extra_edges)},
    {"metric": "approved_and_rejected_conflicts", "count": len(conflicting_edges)},
])

coverage_summary

,metric,count
0,raw_ontology_edges,190
1,excluded_root_top_sense_edges,5
2,reviewable_ontology_edges,185
3,unique_approved_edges,177
4,unique_rejected_edges,32
5,unique_decided_edges,179
6,missing_from_approved_or_rejected,6
7,extra_not_in_ontology,0
8,approved_and_rejected_conflicts,30


In [54]:
# 列出尚未被任何 Approved / Rejected list 覆蓋的 ontology edge。
missing_df = pd.DataFrame(
    sorted(missing_edges),
    columns=["source", "target", "relation"],
)

missing_df

,source,target,relation
0,building,basement,partial
1,building type,educational building,sense
2,event,groundbreaking,sense
3,issue,functionality,sense
4,project phase,feasibility study,sense
5,space,public space,sense


In [55]:
# 列出出現在 checked 結果、但不屬於原 ontology 的 edge；理想狀態應該是空表。
extra_df = pd.DataFrame(
    sorted(extra_edges),
    columns=["source", "target", "relation"],
)

extra_df

,source,target,relation


In [56]:
# ???? edge ??????? approved / rejected????? 2 ?????? decision?
decision_votes = (
    decisions_df[decisions_df["edge_key"].isin(ontology_edges)]
    .groupby(["source", "target", "relation", "decision"], as_index=False)
    .agg(
        file_count=("file", "nunique"),
        files=("file", lambda values: ", ".join(sorted(set(values)))),
    )
)

decision_vote_summary = (
    decision_votes
    .pivot_table(
        index=["source", "target", "relation"],
        columns="decision",
        values="file_count",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

for column in ["approved", "rejected"]:
    if column not in decision_vote_summary.columns:
        decision_vote_summary[column] = 0

vote_file_lists = (
    decision_votes
    .pivot_table(
        index=["source", "target", "relation"],
        columns="decision",
        values="files",
        aggfunc="first",
        fill_value="",
    )
    .reset_index()
    .rename(columns={"approved": "approved_files", "rejected": "rejected_files"})
)

multi_file_decisions_df = (
    decision_vote_summary
    .merge(vote_file_lists, on=["source", "target", "relation"], how="left")
    .query("approved >= 2 or rejected >= 2")
    .sort_values(["approved", "rejected", "source", "target"], ascending=[False, False, True, True])
)

multi_file_decisions_df


decision,source,target,relation,approved,rejected,approved_files,rejected_files
54,envelope,facade,partial,9,0,"于皓-ontology_20.checked-10.checked.dot, 莊書豪-ont...",
7,building,building footprint,data attribute,7,1,"梁庭嘉-ontology_20.checked-5.checked.dot, 莊書豪-ont...",郭瀞淇-ontology_20.checked-3.checked.dot
100,material,Transparency,feature attribute,7,1,"于皓-ontology_20.checked-10.checked.dot, 梁庭嘉-ont...",郭瀞淇-ontology_20.checked-3.checked.dot
13,building,design concept,feature attribute,7,0,"于皓-ontology_20.checked-10.checked.dot, 蔡承軒-ont...",
14,building,envelope,partial,7,0,"于皓-ontology_20.checked-10.checked.dot, 梁庭嘉-ont...",
...,...,...,...,...,...,...,...
113,participant,architect,sense,2,0,"陳宏瑜-ontology_20.checked-7.checked.dot, 陳逸嘉-ont...",
123,participant,user,sense,2,0,"郭瀞淇-ontology_20.checked-3.checked.dot, 陳智偉-ont...",
131,project phase,site analysis,sense,2,0,"莊書豪-ontology_20.checked-8.checked.dot, 謝瑋庭-ont...",
137,site,landscape,feature attribute,2,0,"蘇于庭-ontology_20.checked-2.checked.dot, 謝瑋庭-ont...",


In [57]:
# ???? 2 ??????? rejected ? edge???? source / target / relation?
common_rejected_edges_df = (
    multi_file_decisions_df
    .query("rejected >= 2")
    [["source", "target", "relation", "rejected", "rejected_files"]]
    .sort_values(["rejected", "source", "target"], ascending=[False, True, True])
    .reset_index(drop=True)
)

common_rejected_edges_df


decision,source,target,relation,rejected,rejected_files
0,material,Material Palette,feature attribute,2,"郭瀞淇-ontology_20.checked-3.checked.dot, 陳逸嘉-ont..."
1,urban context,Urban Fabric,feature attribute,2,"蔡承軒-ontology_20.checked-11.checked.dot, 陳宏瑜-on..."


In [58]:
# 列出同一條 edge 同時被不同檔案 approve 與 reject 的情況，方便後續仲裁。
conflict_df = decisions_df[
    decisions_df["edge_key"].isin(conflicting_edges)
].sort_values(["source", "target", "relation", "file"])

conflict_df[["source", "target", "relation", "decision", "file", "reject_reason"]]

,source,target,relation,decision,file,reject_reason
67,building,Spatial Organization,feature attribute,approved,梁庭嘉-ontology_20.checked-5.checked.dot,
188,building,Spatial Organization,feature attribute,approved,蔡承軒-ontology_20.checked-11.checked.dot,
413,building,Spatial Organization,feature attribute,rejected,郭瀞淇-ontology_20.checked-3.checked.dot,
541,building,Spatial Organization,feature attribute,approved,陳智偉-ontology_20.checked-6.checked.dot,
61,building,building footprint,data attribute,approved,梁庭嘉-ontology_20.checked-5.checked.dot,
...,...,...,...,...,...,...
177,user,occupant,sense,approved,莊書豪-ontology_20.checked-8.checked.dot,
532,user,occupant,sense,rejected,陳宏瑜-ontology_20.checked-7.checked.dot,
58,zoning,setback,feature attribute,approved,于皓-ontology_20.checked-10.checked.dot,
115,zoning,setback,feature attribute,approved,梁庭嘉-ontology_20.checked-5.checked.dot,


## 目前初步結果

目前 coverage 計算會排除 root `design case` 的 5 條 `top sense` 骨架 edge，讓 denominator 對齊 ontology checker 的 Check List / Left 計算。